In [13]:
import os
from os import path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import shapiro
from google.colab import drive
from google.colab import data_table


drive.mount('/content/drive')
data_table.enable_dataframe_formatter()
file_path = "/content/drive/MyDrive/Virtual Reality Lesson and Community Smells/Questionari/csv/"
file_name = "dataset_for_test.csv"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
# dataset loading
df = pd.read_csv(file_path + file_name)
df_clean = df.iloc[1:].reset_index(drop=True)

modes = df_clean.iloc[:, -1]

# column mapping
variables = {
    "LO": list(range(0, 13)),
    "SEff": [13, 14, 15, 16, 17, 18, 19, 20, 22],
    "IM": [24, 25, 26, 28, 29, 30, 31, 32],
    "AE": list(range(33, 38)),
    "BEc": list(range(38, 40)),
    "BEe": [41, 42, 43],
    "DEr": list(range(49, 52)),
    "PW": list(range(52, 58)),
    "ST": [58],
}

df_scores = pd.DataFrame()
df_scores["Group"] = modes

for var_name, col_indices in variables.items():
  if var_name == "LO":
    df_scores[var_name] = (
        df_clean.iloc[:, col_indices].astype(float).sum(axis=1)
    )
  else:
    df_scores[var_name] = (
        df_clean.iloc[:, col_indices].astype(float).median(axis=1)
    )

df_scores.head(10)

,Group,LO,SEff,IM,AE,BEc,BEe,DEr,PW,ST
0,Presence,13.0,5.0,5.0,6.0,7.0,6.0,7.0,10.0,10.0
1,Presence,13.0,4.0,3.0,5.0,7.0,5.0,7.0,7.5,8.0
2,Presence,12.0,3.0,5.0,6.0,7.0,7.0,6.0,9.0,8.0
3,Presence,11.0,4.0,5.0,7.0,7.0,4.0,7.0,10.0,8.0
4,Presence,12.0,4.0,4.5,6.0,7.0,5.0,7.0,9.0,10.0
5,Presence,12.0,5.0,5.0,7.0,7.0,7.0,6.0,9.0,10.0
6,Presence,13.0,4.0,4.0,5.0,6.5,5.0,6.0,10.0,9.0
7,Presence,13.0,3.0,3.0,5.0,6.0,1.0,7.0,7.5,7.0
8,Presence,13.0,4.0,4.0,6.0,6.5,7.0,7.0,7.5,8.0
9,Presence,13.0,4.0,4.5,6.0,6.0,6.0,4.0,6.5,8.0


In [15]:
groups = ["Presence", "Remote", "VR"]
shapiro_results = []

for var in variables.keys():
  for grp in groups:
    data_subset = df_scores[df_scores["Group"] == grp][var]

    # Shapiro-Wilk
    stat_w, p_value = shapiro(data_subset)

    # check p < 0.05
    is_normal = "Yes" if p_value > 0.05 else "No (Non-normal)"

    shapiro_results.append({
        "Variable": var,
        "Group": grp,
        "W-Statistic": round(stat_w, 4),
        "p-value": round(p_value, 4),
        "Normal (p > 0.05)": is_normal,
    })

df_shapiro_results = pd.DataFrame(shapiro_results)
df_shapiro_results

,Variable,Group,W-Statistic,p-value,Normal (p > 0.05)
0,LO,Presence,0.7309,0.0021,No (Non-normal)
1,LO,Remote,0.6785,0.0005,No (Non-normal)
2,LO,VR,0.8290,0.0325,No (Non-normal)
3,SEff,Presence,0.8148,0.0219,No (Non-normal)
4,SEff,Remote,0.8325,0.0359,No (Non-normal)
5,SEff,VR,0.8022,0.0154,No (Non-normal)
6,IM,Presence,0.8197,0.0251,No (Non-normal)
7,IM,Remote,0.6988,0.0009,No (Non-normal)
8,IM,VR,0.8871,0.1574,Yes
9,AE,Presence,0.8325,0.0359,No (Non-normal)


In [16]:
pivot_w = df_shapiro_results.pivot(
    index="Variable", columns="Group", values="W-Statistic"
)
pivot_p = df_shapiro_results.pivot(
    index="Variable", columns="Group", values="p-value"
)

summary_table = pd.DataFrame()
for grp in groups:
  summary_table[f"{grp} (W)"] = pivot_w[grp]
  summary_table[f"{grp} (p)"] = pivot_p[grp]

order = ["LO", "SEff", "IM", "AE", "BEc", "BEe", "DEr", "PW", "ST"]
summary_table = summary_table.reindex(order)

print("--- SHAPIRO-WILK TABLE ---")
display(summary_table)

summary_table.to_csv(file_path + "shapiro_wilk_results.csv")
print("\nFile 'shapiro_wilk_results.csv' saved")

--- SHAPIRO-WILK TABLE ---


,Presence (W),Presence (p),Remote (W),Remote (p),VR (W),VR (p)
Variable,,,,,,
LO,0.7309,0.0021,0.6785,0.0005,0.8290,0.0325
SEff,0.8148,0.0219,0.8325,0.0359,0.8022,0.0154
IM,0.8197,0.0251,0.6988,0.0009,0.8871,0.1574
AE,0.8325,0.0359,0.5316,0.0000,0.3657,0.0000
BEc,0.7172,0.0014,0.8790,0.1272,0.7708,0.0064
BEe,0.8375,0.0412,0.8877,0.1599,0.9065,0.2576
DEr,0.6785,0.0005,0.6497,0.0002,0.8242,0.0285
PW,0.8754,0.1155,0.8365,0.0400,0.9374,0.5248
ST,0.8258,0.0298,0.6262,0.0001,0.6938,0.0007



File 'shapiro_wilk_results.csv' saved
